In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr
import sys
sys.path.insert(0, '/home/codycruz/LLC_ocean_emulator/high_res_diagnostics')
from predictions_io import open_predictions
import cftime
import os

In [3]:
# ============== LOAD LLC ==============

# DOMAIN
i_extent=slice(2880,3600)
i_g_extent=slice(2880,3600)
j_extent=slice(720,1440)
j_g_extent=slice(720,1440)

llc_patch_full = xr.open_zarr('/orcd/data/abodner/003/LLC4320/LLC4320',consolidated=False).isel(
    face=1,
    i=i_extent,
    i_g=i_g_extent,
    j=j_extent,
    j_g=j_g_extent
)

# ============== LOAD EMULATORS ==============
emulator_configs = [
    {
        'name': '1-face-ckpt6',
        'key': 'emulator_1',
        'path': '/orcd/pool/008/codycruz/inference_patch/1-face/epoch6-ema-quintic-TRAIN-test-25323691/predictions.zarr',
        'desc': ''
    },
]

# ============== OPEN EMULATOR DATASETS ==============
emulator_patches_raw = {}
for cfg in emulator_configs:
    # Load emulator data
    emu_ds = open_predictions(cfg['path'])
    
    # Rename dimensions to match LLC if needed
    # (adjust these names based on your actual emulator dimension names)
    dim_mapping = {}
    if 'lat' in emu_ds.dims and 'j' not in emu_ds.dims:
        dim_mapping['lat'] = 'j'
    if 'lon' in emu_ds.dims and 'i' not in emu_ds.dims:
        dim_mapping['lon'] = 'i'
    
    if dim_mapping:
        emu_ds = emu_ds.rename(dim_mapping)
    
    # Apply same spatial slicing as LLC
    emu_ds = emu_ds.isel(i=i_extent, j=j_extent)
    
    # Align emulator to LLC's coordinates
    emu_ds = emu_ds.assign_coords({
        'i': llc_patch_full.i,
        'j': llc_patch_full.j
    })
    
    emulator_patches_raw[cfg['key']] = emu_ds
    print(f"Loaded {cfg['name']}: {cfg['desc']}")

# ============== TIME MATCHING ==============
def normalize_times(times):
    return pd.DatetimeIndex([
        pd.Timestamp(
            int(t.year), int(t.month), int(t.day),
            int(t.hour), int(t.minute), int(t.second)
        )
        if hasattr(t, "year")
        else pd.Timestamp(t).floor("s")
        for t in times
    ])

llc_times_norm = normalize_times(llc_patch_full.time.values)

common_times = llc_times_norm
for cfg in emulator_configs:
    emulator_times_norm = normalize_times(emulator_patches_raw[cfg['key']].time.values)
    common_times = common_times.intersection(emulator_times_norm)

common_times = common_times.sort_values()

llc_mask = llc_times_norm.isin(common_times)
llc_patch = llc_patch_full.isel(time=llc_mask)

print(f"LLC subset to {len(common_times)} common times")

# ============== PORT GRID VARS & BUILD UNIFIED STRUCTURE ==============
grid_vars = ['XC', 'YC', 'rA', 'Z', 'dxC', 'dyC', 'dxG', 'dyG']

emulator_patches = {}
for cfg in emulator_configs:
    patch_raw = emulator_patches_raw[cfg['key']]
    patch_times_norm = normalize_times(patch_raw.time.values)

    patch_mask = patch_times_norm.isin(common_times)
    patch = patch_raw.isel(time=patch_mask)

    # Add LLC grid variables
    for gv in grid_vars:
        if gv in llc_patch:
            patch[gv] = llc_patch[gv]

    emulator_patches[cfg['key']] = patch

# ============== UNIFIED REFERENCE LISTS ==============
emulator_info = [(cfg['name'], cfg['key']) for cfg in emulator_configs]
n_emulators = len(emulator_info)

all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"\n=== Setup complete: LLC + {n_emulators} emulators ===")
for name, key in emulator_info:
    print(f"  {name} ({key})")
    print(f"    Shape: {emulator_patches[key][list(emulator_patches[key].data_vars)[0]].shape}")
    print(f"    i range: [{emulator_patches[key].i.values.min()}, {emulator_patches[key].i.values.max()}]")
    print(f"    j range: [{emulator_patches[key].j.values.min()}, {emulator_patches[key].j.values.max()}]")

print(f"LLC shape: {llc_patch[list(llc_patch.data_vars)[0]].shape}")

Loaded 1-face-ckpt6: 
LLC subset to 83 common times

=== Setup complete: LLC + 1 emulators ===
  1-face-ckpt6 (emulator_1)
    Shape: (83, 51, 720, 720)
    i range: [2880, 3599]
    j range: [720, 1439]
LLC shape: (720, 720)


In [4]:
selected_time_range = [0, 72]   # inclusive indices
stepping = 18                    # 1 = every timestep, 4 = every 4th timestep

start_idx, end_idx = selected_time_range

# ----------------------------------------
# First subset LLC
# ----------------------------------------
llc_patch = llc_patch.isel(
    time=slice(start_idx, end_idx + 1, stepping)
)

# ----------------------------------------
# Then subset each emulator safely
# Handles shorter emulator runs automatically
# ----------------------------------------
emulator_patches_subset = {}

for key, patch in emulator_patches.items():

    max_time = patch.sizes['time']

    # Prevent indexing past emulator length
    safe_end_idx = min(end_idx, max_time - 1)

    patch_subset = patch.isel(
        time=slice(start_idx, safe_end_idx + 1, stepping)
    )

    emulator_patches_subset[key] = patch_subset

emulator_patches = emulator_patches_subset

# ----------------------------------------
# Match LLC length to shortest emulator
# ----------------------------------------
min_time_len = min(
    [llc_patch.sizes['time']] +
    [patch.sizes['time'] for patch in emulator_patches.values()]
)

llc_patch = llc_patch.isel(time=slice(0, min_time_len))

emulator_patches = {
    key: patch.isel(time=slice(0, min_time_len))
    for key, patch in emulator_patches.items()
}

# ----------------------------------------
# Rebuild combined dict
# ----------------------------------------
all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

# ----------------------------------------
# Diagnostics
# ----------------------------------------
print(f"Subset to time indices {start_idx}:{end_idx}")
print(f"Stepping = {stepping}")
print(f"Final synchronized length = {min_time_len}")

print(f"LLC now has {llc_patch.sizes['time']} times")

for name, key in emulator_info:
    print(
        f"{name} ({key}) now has "
        f"{emulator_patches[key].sizes['time']} times"
    )

Subset to time indices 0:72
Stepping = 18
Final synchronized length = 5
LLC now has 5 times
1-face-ckpt6 (emulator_1) now has 5 times


In [5]:
def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"

In [6]:
# ============== VERTICAL-AXIS HELPERS (W lives on cell interfaces) ==============
# LLC keeps W on `k_p1` -- 52 cell INTERFACES -- while Theta/Salt/U/V sit on `k`,
# 51 cell CENTRES. The emulator writes W back onto `k` with 51 levels. So a
# blanket `.isel(k=...)` silently leaves LLC's W at full depth (the level index
# never applies) and the plot then fails on a 3-D array, or a difference
# broadcasts 52 against 51. Level L means the same layer in both: the emulator's
# `k=L` is trained against LLC's `k_p1=L`.
VERTICAL_DIMS = ("k", "k_p1", "k_l", "k_u", "lev")


def vertical_dim_of(da):
    """Whichever vertical dimension this variable carries, or None if surface."""
    for dim in VERTICAL_DIMS:
        if dim in da.dims:
            return dim
    return None


def pick(patch, var, *, time=None, k=None):
    """`patch[var]` with `time`/`k` applied to the axes it actually has."""
    da = patch[var]
    if time is not None:
        da = da.isel(time=time)
    if k is not None:
        dim = vertical_dim_of(da)
        if dim is not None:
            da = da.isel({dim: k})
    return da


def common_levels(*patch_var_pairs):
    """Levels the given (patch, var) pairs share.

    LLC's W has 52 and the emulator's has 51, so depth comparisons have to trim
    to the shorter of the two rather than broadcast.
    """
    sizes = []
    for patch, var in patch_var_pairs:
        dim = vertical_dim_of(patch[var])
        sizes.append(patch[var].sizes[dim] if dim is not None else 1)
    return min(sizes)


def depth_axis(patch, var):
    """Depth in metres for each level of `var`, negative downward.

    The store carries `Z` for cell centres but no coordinate for the interfaces
    W sits on, so derive them: face[0] = 0, face[i+1] = 2*centre[i] - face[i].
    That is the same construction as DEPTH_INTERFACES in ocean_emulators.
    """
    centres = np.asarray(patch['Z'].values, dtype=float)
    if vertical_dim_of(patch[var]) != 'k_p1':
        return centres
    faces = [0.0]
    for centre in centres:
        faces.append(2.0 * centre - faces[-1])
    return np.asarray(faces)


# surface field and field difference

In [7]:
# ============== SET VARIABLES HERE ==============
# prog_vars = ['Theta', 'W', 'U']#, 'Salt', 'U', 'V']
# colormaps = {'Theta': 'Spectral_r', 'W': 'magma', 'U': 'bwr'}#, 'Salt': 'viridis', 'U': 'bwr', 'V': 'bwr'}

prog_vars = ['Theta']#, 'Salt', 'U', 'V']
colormaps = {'Theta': 'Spectral_r'}#, 'Salt': 'viridis', 'U': 'bwr', 'V': 'bwr'}
# ================================================
for var in prog_vars:
    print(f"Generating plots for {var}...")
    
    os.makedirs(f'figs/simple/', exist_ok=True)
    
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    
    # (i_extent, j_extent, i_g_extent, j_g_extent) for each zoom subrow
    domains = [
        (slice(2880,3600), slice(720,1440), slice(2880,3600), slice(720,1440)),
        (slice(2880+100,2880+300), slice(720+100,720+300), slice(2880+100,2880+300), slice(720+100,720+300)),
        (slice(2880+100+25,2880+100+25+50), slice(720+100+75,720+100+75+50), slice(2880+100+25,2880+100+25+50), slice(720+100+75,720+100+75+50)),
    ]
    n_sub = len(domains)
    
    nrows = len(time_indices) * n_sub   # 3 rows per time
    ncols_fields = 1 + n_emulators  # LLC + emulators
    ncols_diff = n_emulators         # emulators only
    
    cmap = colormaps[var]
    
    # ==================== PLOT 1: Surface fields ====================
    fig, axes = plt.subplots(nrows, ncols_fields, figsize=(3.6*ncols_fields, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    if ncols_fields == 1:
        axes = axes.reshape(-1, 1)
    
    for t_idx, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        for d_idx, (i_ext, j_ext, i_g_ext, j_g_ext) in enumerate(domains):
            row = t_idx * n_sub + d_idx
            
            # Collect all surface fields for this subrow
            fields = [pick(llc_patch, var, time=t, k=0).sel(i=i_ext, j=j_ext)]
            for emu_name, emu_key in emulator_info:
                fields.append(pick(emulator_patches[emu_key], var, time=t, k=0).sel(i=i_g_ext, j=j_g_ext))
            
            # Calculate vmin/vmax for this subrow only
            finite_fields = [f.values[np.isfinite(f.values)] for f in fields]
            if not any(vals.size for vals in finite_fields):
                print(f"  Skipping {var} {time_str} zoom {d_idx+1}: no finite values")
                continue
            vmin = np.min([vals.min() for vals in finite_fields if vals.size])
            vmax = np.max([vals.max() for vals in finite_fields if vals.size])
            
            labels = ['LLC'] + [name for name, _ in emulator_info]
            
            cf = None
            for col, (field, label) in enumerate(zip(fields, labels)):
                ax = axes[row, col]
                field_values = np.ma.masked_invalid(field.values)
                cf = ax.contourf(field.coords.get('i', np.arange(field.shape[-1])),
                                 field.coords.get('j', np.arange(field.shape[-2])), field_values,
                                 cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
                ax.set_title(f'{label} {var} {time_str} zoom{d_idx+1}', fontsize=8)
            
            # Add one colorbar for this subrow (shared by all columns)
            cbar = fig.colorbar(cf, ax=axes[row, :], pad=0.02, fraction=0.046)

    plt.savefig(f'figs/simple/surface-zoom-{var}.png')
    plt.close()
    
    print(f"✓ Saved plots for {var}")

Generating plots for Theta...
✓ Saved plots for Theta
